# Streaming Data Load (Leuschner SDR)

Load raw per-dump `.npz` files from `data/lab04/streaming/m31/` and assemble
them into the array format used by analysis notebooks.

Each streaming file contains a single dump with keys:
- `corr00` — auto-correlation pol 0, real float32 (1024 channels)
- `corr01` — cross-correlation pol 0×1, complex64 (1024 channels)
- `corr11` — auto-correlation pol 1, real float32 (1024 channels)
- `lo_freq_mhz` — LO frequency (1420 or 1421 MHz)
- `time` — Unix timestamp (seconds, scalar)
- `target_name`, `alt_deg`, `az_deg`, `ra_deg`, `dec_deg` — pointing
- `seq` — sequence number within the observation

Dumps alternate between LO = 1420 MHz ("ON") and LO = 1421 MHz ("OFF")
for frequency-switching bandpass removal.

The data collection script (`observe.py`) also records a **noise diode
calibration** at the start of the observation:
`data/lab04/streaming/calibration/noisecal_*.npz`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# SDR / observation parameters
SAMPLE_RATE_HZ = 2.56e6
NFFT           = 1024
LO_ON_MHZ     = 1420.0
LO_OFF_MHZ    = 1421.0

%matplotlib inline
plt.rcParams.update({"figure.dpi": 130, "figure.facecolor": "white"})
print("Imports OK.")

## 1. Discover and load raw dumps

Each file is one SDR capture dump (~0.8 s cadence). We sort by filename
(which embeds a timestamp and sequence number) and stack into arrays,
separating ON (LO = 1420 MHz) and OFF (LO = 1421 MHz) dumps.

In [ ]:
DATA_DIR = Path("../../../data/lab04/streaming/m31")
dump_paths = sorted(DATA_DIR.glob("m31_dump_*.npz"))
N_dumps = len(dump_paths)
print(f"Found {N_dumps} dump files in {DATA_DIR}")

# Peek at first file for array dimensions
with np.load(dump_paths[0], allow_pickle=True) as first:
    N_CH = first["corr01"].shape[0]
    print(f"Channels per dump: {N_CH}")

# Pre-allocate
corr00_raw   = np.empty((N_dumps, N_CH), dtype=float)
corr01_raw   = np.empty((N_dumps, N_CH), dtype=complex)
corr11_raw   = np.empty((N_dumps, N_CH), dtype=float)
lo_mhz_arr   = np.empty(N_dumps, dtype=float)
unix_arr      = np.empty(N_dumps, dtype=float)
alt_deg_arr   = np.empty(N_dumps, dtype=float)
az_deg_arr    = np.empty(N_dumps, dtype=float)
ra_deg_arr    = np.empty(N_dumps, dtype=float)
dec_deg_arr   = np.empty(N_dumps, dtype=float)
seq_arr       = np.empty(N_dumps, dtype=int)

for i, path in enumerate(dump_paths):
    with np.load(path, allow_pickle=True) as npz:
        corr00_raw[i]  = npz["corr00"]
        corr01_raw[i]  = npz["corr01"]
        corr11_raw[i]  = npz["corr11"]
        lo_mhz_arr[i]  = float(npz["lo_freq_mhz"])
        unix_arr[i]    = float(npz["time"])
        alt_deg_arr[i] = float(npz["alt_deg"])
        az_deg_arr[i]  = float(npz["az_deg"])
        ra_deg_arr[i]  = float(npz["ra_deg"])
        dec_deg_arr[i] = float(npz["dec_deg"])
        seq_arr[i]     = int(npz["seq"])
    if (i + 1) % 200 == 0:
        print(f"  loaded {i + 1}/{N_dumps}")

print(f"Loaded {N_dumps} dumps, shape = {corr01_raw.shape}")

# Separate ON and OFF dumps
on_mask  = lo_mhz_arr == LO_ON_MHZ
off_mask = lo_mhz_arr == LO_OFF_MHZ
N_on  = on_mask.sum()
N_off = off_mask.sum()
print(f"ON  dumps (LO={LO_ON_MHZ} MHz): {N_on}")
print(f"OFF dumps (LO={LO_OFF_MHZ} MHz): {N_off}")

## 1b. Load noise diode calibration

The observation script collects per-polarisation auto- and cross-correlations
with the noise diode ON at both LO frequencies. This provides:
- $T_{\rm noise}$ in correlator units (noise ON − noise OFF baseline)
- Bandpass shape at each LO setting

In [ ]:
CAL_DIR = Path("../../../data/lab04/streaming/calibration")
cal_path = sorted(CAL_DIR.glob("noisecal_*.npz"))[-1]

with np.load(cal_path, allow_pickle=True) as cal:
    # Noise ON at LO = 1420 MHz
    cal_corr00_on = np.asarray(cal["corr00_on"], dtype=float)
    cal_corr01_on = np.asarray(cal["corr01_on"], dtype=complex)
    cal_corr11_on = np.asarray(cal["corr11_on"], dtype=float)
    cal_times_on  = np.asarray(cal["times_on"], dtype=float)
    # Noise ON at LO = 1421 MHz
    cal_corr00_off = np.asarray(cal["corr00_off"], dtype=float)
    cal_corr01_off = np.asarray(cal["corr01_off"], dtype=complex)
    cal_corr11_off = np.asarray(cal["corr11_off"], dtype=float)
    cal_times_off  = np.asarray(cal["times_off"], dtype=float)

# Time-averaged calibration spectra (noise ON)
cal_corr00_on_mean = np.nanmean(cal_corr00_on, axis=0)
cal_corr01_on_mean = np.nanmean(cal_corr01_on, axis=0)
cal_corr11_on_mean = np.nanmean(cal_corr11_on, axis=0)

cal_corr00_off_mean = np.nanmean(cal_corr00_off, axis=0)
cal_corr01_off_mean = np.nanmean(cal_corr01_off, axis=0)
cal_corr11_off_mean = np.nanmean(cal_corr11_off, axis=0)

print(f"Loaded calibration: {cal_path.name}")
print(f"  LO={LO_ON_MHZ} MHz: {cal_corr00_on.shape[0]} dumps, {cal_corr00_on.shape[1]} channels")
print(f"  LO={LO_OFF_MHZ} MHz: {cal_corr00_off.shape[0]} dumps, {cal_corr00_off.shape[1]} channels")

## 2. Derive auxiliary quantities

Compute the frequency axis, Stokes parameters, and cadence statistics.

In [ ]:
# ── Frequency axis (baseband channels) ──────────────────────────────
DF_HZ = SAMPLE_RATE_HZ / NFFT                          # 2500 Hz per channel
F_BASEBAND_KHZ = np.arange(N_CH) * DF_HZ / 1e3         # 0 to 2560 kHz
F_BASEBAND_MHZ = F_BASEBAND_KHZ / 1e3

# Sky frequency depends on LO setting:
#   f_sky = LO + f_baseband
# For ON dumps:  1420.0 + [0, 2.56) MHz
# For OFF dumps: 1421.0 + [0, 2.56) MHz

# ── Cadence ─────────────────────────────────────────────────────────
dt = np.diff(unix_arr)
median_cadence = float(np.median(dt))

# ── Stokes parameters (from all dumps) ─────────────────────────────
# I = corr00 + corr11  (total intensity)
# Q = corr00 - corr11
# U = 2 * Re(corr01)
# V = 2 * Im(corr01)
stokes_I = corr00_raw + corr11_raw
stokes_Q = corr00_raw - corr11_raw
stokes_U = 2.0 * corr01_raw.real
stokes_V = 2.0 * corr01_raw.imag

print(f"Baseband channels: {N_CH}, Δν = {DF_HZ:.0f} Hz = {DF_HZ/1e3:.2f} kHz")
print(f"Velocity resolution: Δv = c·Δν/f = {3e5 * DF_HZ / (1420.405e6):.2f} km/s")
print(f"Median cadence: {median_cadence:.3f} s")
print(f"Time span: {(unix_arr[-1] - unix_arr[0]) / 3600:.2f} hours")

## 3. Frequency-switched bandpass removal

Divide ON by OFF to remove the instrumental bandpass shape. The sky
signal shifts by 1 MHz between the two LO settings, but the bandpass
(SDR + RF chain frequency response) stays fixed in channel space.

We pair consecutive ON/OFF dumps by index (they alternate 1:1) and
compute `ON / OFF` for Stokes I.

In [ ]:
# ── Pair ON and OFF dumps ───────────────────────────────────────────
on_indices  = np.where(on_mask)[0]
off_indices = np.where(off_mask)[0]

# Pair by position: dump 0 (ON) with dump 1 (OFF), dump 2 (ON) with dump 3 (OFF), etc.
n_pairs = min(len(on_indices), len(off_indices))
on_idx  = on_indices[:n_pairs]
off_idx = off_indices[:n_pairs]

# Stokes I for ON and OFF
I_on  = stokes_I[on_idx]   # (n_pairs, N_CH)
I_off = stokes_I[off_idx]  # (n_pairs, N_CH)

# Frequency-switched ratio: (ON - OFF) / OFF
# This isolates the spectral line signal from the bandpass
fs_ratio = (I_on - I_off) / I_off  # (n_pairs, N_CH)

# Timestamps for paired dumps (midpoint)
unix_paired = 0.5 * (unix_arr[on_idx] + unix_arr[off_idx])

print(f"Paired {n_pairs} ON/OFF dump pairs")
print(f"Frequency-switched ratio shape: {fs_ratio.shape}")

# Time-averaged frequency-switched spectrum
fs_mean = np.nanmean(fs_ratio, axis=0)
fs_std  = np.nanstd(fs_ratio, axis=0) / np.sqrt(n_pairs)

print(f"Mean ratio range: [{np.nanmin(fs_mean):.4f}, {np.nanmax(fs_mean):.4f}]")

## 4. Raw spectra and frequency-switched spectrum

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True,
                         gridspec_kw={"hspace": 0.08})

# (a) Mean Stokes I — ON and OFF
I_on_mean  = np.nanmean(I_on, axis=0)
I_off_mean = np.nanmean(I_off, axis=0)

axes[0].plot(F_BASEBAND_MHZ, I_on_mean, lw=0.5, alpha=0.7, label=f"ON (LO={LO_ON_MHZ})")
axes[0].plot(F_BASEBAND_MHZ, I_off_mean, lw=0.5, alpha=0.7, label=f"OFF (LO={LO_OFF_MHZ})")
axes[0].set_ylabel("Stokes I [counts]")
axes[0].set_title("Time-averaged spectra")
axes[0].legend(fontsize=8)

# (b) Mean auto-correlations (pol 0 and pol 1)
axes[1].plot(F_BASEBAND_MHZ, np.nanmean(corr00_raw[on_mask], axis=0),
             lw=0.5, alpha=0.7, label="corr00 (pol 0)")
axes[1].plot(F_BASEBAND_MHZ, np.nanmean(corr11_raw[on_mask], axis=0),
             lw=0.5, alpha=0.7, label="corr11 (pol 1)")
axes[1].set_ylabel("Auto-correlation [counts]")
axes[1].legend(fontsize=8)

# (c) Frequency-switched spectrum
axes[2].plot(F_BASEBAND_MHZ, fs_mean, lw=0.5, color="C0", alpha=0.8)
axes[2].fill_between(F_BASEBAND_MHZ, fs_mean - fs_std, fs_mean + fs_std,
                     alpha=0.15, color="C0")
axes[2].axhline(0, color="gray", lw=0.5, ls="--")
axes[2].set_ylabel("(ON − OFF) / OFF")
axes[2].set_xlabel("Baseband frequency [MHz]")

fig.tight_layout()
plt.show()

## 5. Observation timeline and cadence

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 5.5), sharex=True,
                         gridspec_kw={"hspace": 0.08})

elapsed_min = (unix_arr - unix_arr[0]) / 60.0

# (a) Altitude
axes[0].scatter(elapsed_min, alt_deg_arr, s=0.3, color="C0", alpha=0.5, rasterized=True)
axes[0].set_ylabel("Altitude [deg]")
axes[0].set_title("Observation timeline")

# (b) Inter-dump cadence
axes[1].scatter(elapsed_min[:-1], dt, s=0.3, color="C0", alpha=0.5, rasterized=True)
axes[1].set_ylabel("Cadence [s]")
axes[1].axhline(median_cadence, color="gray", lw=0.5, ls="--",
                label=f"median = {median_cadence:.2f} s")
axes[1].legend(fontsize=8)

# (c) Cumulative dumps, coloured by LO
axes[2].scatter(elapsed_min[on_mask], np.cumsum(on_mask)[on_mask],
                s=0.3, color="C0", alpha=0.5, label="ON", rasterized=True)
axes[2].scatter(elapsed_min[off_mask], np.cumsum(off_mask)[off_mask],
                s=0.3, color="C1", alpha=0.5, label="OFF", rasterized=True)
axes[2].set_ylabel("Cumulative dumps")
axes[2].set_xlabel("Elapsed time [min]")
axes[2].legend(fontsize=8)

fig.tight_layout()
plt.show()

print(f"Total dumps:     {N_dumps} ({N_on} ON + {N_off} OFF)")
print(f"Median cadence:  {median_cadence:.3f} s")
print(f"Alt range:       {alt_deg_arr.min():.1f} to {alt_deg_arr.max():.1f} deg")

## 6. Duty cycle

The duty cycle is the fraction of wall-clock time spent integrating.
Each dump integrates `nblocks − 1 = 64` blocks of `nsamples / sample_rate`
seconds; the rest is overhead (USB transfer, FFT, correlation, disk I/O).

In [ ]:
NSAMPLES = 32768
NBLOCKS_VALID = 64  # nblocks=65 minus block 0

# Integration time per dump
t_int_per_dump = NBLOCKS_VALID * (NSAMPLES / SAMPLE_RATE_HZ)  # seconds

# Wall-clock cadence between consecutive dumps
wall_cadence = np.diff(unix_arr)

# Per-gap duty cycle
duty_per_gap = t_int_per_dump / wall_cadence

# Overall
total_int_time  = N_dumps * t_int_per_dump
total_wall_time = float(unix_arr[-1] - unix_arr[0])
overall_duty    = total_int_time / total_wall_time

fig, ax = plt.subplots(figsize=(10, 2.8))
ax.scatter(elapsed_min[:-1], duty_per_gap * 100, s=0.3, color="C0",
           alpha=0.5, rasterized=True)
ax.axhline(overall_duty * 100, color="gray", lw=0.5, ls="--",
           label=f"overall = {overall_duty * 100:.1f}%")
ax.set_ylabel("Duty cycle [%]")
ax.set_xlabel("Elapsed time [min]")
ax.set_ylim(0, 105)
ax.set_title("Duty cycle")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

print(f"Integration per dump:   {t_int_per_dump:.3f} s")
print(f"Total integration time: {total_int_time:.1f} s")
print(f"Total wall-clock time:  {total_wall_time:.1f} s")
print(f"Overall duty cycle:     {overall_duty * 100:.1f}%")

## 7. Summary statistics

In [ ]:
import datetime as _dt

t0 = _dt.datetime.fromtimestamp(unix_arr[0], tz=_dt.UTC)
t1 = _dt.datetime.fromtimestamp(unix_arr[-1], tz=_dt.UTC)

print(f"Observation start (UTC): {t0:%Y-%m-%d %H:%M:%S}")
print(f"Observation end   (UTC): {t1:%Y-%m-%d %H:%M:%S}")
print(f"Duration:                {total_wall_time / 60:.1f} min")
print(f"Dumps:                   {N_dumps} ({N_on} ON + {N_off} OFF)")
print(f"Paired ON/OFF:           {n_pairs}")
print(f"Channels:                {N_CH} (Δν = {DF_HZ/1e3:.2f} kHz)")
print(f"LO frequencies:          {LO_ON_MHZ}, {LO_OFF_MHZ} MHz")
print(f"Median cadence:          {median_cadence:.3f} s")
print(f"Integration per dump:    {t_int_per_dump:.3f} s")
print(f"Duty cycle:              {overall_duty:.1%}")
print(f"Alt range:               {alt_deg_arr.min():.1f} to {alt_deg_arr.max():.1f} deg")
print(f"Target RA:               {ra_deg_arr[0]:.4f} deg")
print(f"Target Dec:              {dec_deg_arr[0]:.4f} deg")